In [1]:
!pip install -q transformers gradio torch

In [3]:
# 1. IMPORT LIBRARIES
import gradio as gr
from transformers import pipeline
import torch

# Check for GPU acceleration
device = 0 if torch.cuda.is_available() else -1
print(f"Loading AI Models on: {'GPU' if device == 0 else 'CPU'}...")

# 2. INITIALIZE INTENT, SENTIMENT, & RESPONDER PIPELINES
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)

# FIX: Using 'text-generation' with an instruction-tuned model that supports it
print("Loading text generation model...")
responder = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=device,
    torch_dtype=torch.bfloat16 if device == 0 else torch.float32
)

# 3. THE BACKEND LOGIC
def process_support_ticket(ticket_text, priority_override):
    if not ticket_text.strip():
        return "Please enter a support ticket message.", "N/A", "N/A", ""

    # Task A: Detect Sentiment
    sentiment_result = sentiment_analyzer(ticket_text)[0]
    sentiment_label = sentiment_result['label']

    # Task B: Categorize the Issue (Zero-Shot)
    candidate_labels = ["Billing & Invoice", "Technical Support", "Shipping & Delivery", "Account Access"]
    classification = classifier(ticket_text, candidate_labels=candidate_labels)
    top_category = classification['labels'][0]

    # Determine Priority based on sentiment and override
    if priority_override != "Auto-Detect":
        priority = priority_override
    else:
        priority = "🔴 HIGH" if sentiment_label == "NEGATIVE" else "🟢 LOW"

    # Task C: Generate Auto-Response Draft with TinyLlama Chat Template
    messages = [
        {"role": "system", "content": f"You are an AI Customer Support assistant. Draft a short, professional response to the customer's issue. Category: {top_category}."},
        {"role": "user", "content": f"Issue: {ticket_text}"}
    ]

    # Format the prompt using the model's standard chat format
    prompt = responder.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    response_generation = responder(prompt, max_new_tokens=120, do_sample=True, temperature=0.7, top_k=50)

    # Extract only the newly generated text response
    full_text = response_generation[0]['generated_text']
    email_draft = full_text.split("<|assistant|>")[-1].strip()

    return top_category, sentiment_label, priority, email_draft

# 4. BUILD THE GRADIO DASHBOARD UI
print("Launching AI Support Desk Interface...")

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 🤖 SmartSupport: AI Customer Ticket Desk")
    gr.Markdown("Drop an incoming customer complaint below. The AI will instantly classify the department, analyze urgency, and write a customized email response.")

    with gr.Row():
        with gr.Column():
            ticket_input = gr.Textbox(
                label="Incoming Customer Email / Ticket",
                placeholder="Type or paste customer complaint here...",
                lines=6
            )

            priority_dropdown = gr.Dropdown(
                choices=["Auto-Detect", "🔴 HIGH", "🟡 MEDIUM", "🟢 LOW"],
                value="Auto-Detect",
                label="Priority Override (Optional)"
            )

            submit_btn = gr.Button("Analyze & Draft Response", variant="primary")

            gr.Examples(
                examples=[
                    ["My order #1042 was supposed to arrive two days ago but the tracking link is broken. I need this package by Friday!", "Auto-Detect"],
                    ["I was charged twice on my credit card for this month's subscription. Please refund the duplicate charge immediately.", "Auto-Detect"],
                    ["Your website keeps crashing every time I try to click the checkout button. I tried clearing my cache but it didn't help.", "Auto-Detect"]
                ],
                inputs=[ticket_input, priority_dropdown]
            )

        with gr.Column():
            with gr.Row():
                category_output = gr.Textbox(label="Detected Category", interactive=False)
                sentiment_output = gr.Textbox(label="Customer Sentiment", interactive=False)
                priority_output = gr.Textbox(label="Assigned Priority Status", interactive=False)

            email_output = gr.Textbox(
                label="AI-Generated Email Draft Reply",
                lines=8,
                interactive=False
            )

    submit_btn.click(
        fn=process_support_ticket,
        inputs=[ticket_input, priority_dropdown],
        outputs=[category_output, sentiment_output, priority_output, email_output]
    )



Loading AI Models on: GPU...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading text generation model...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Launching AI Support Desk Interface...


/tmp/ipykernel_1020/3483056033.py:63: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:


In [4]:
demo.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://277aaf23461e51bbb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
